# HybridMixtureNetwork â€” Intraday 30min + Classification + Optuna

Single TCN + MDN with VAE regime autoencoder and MLP head,
adapted for **intraday 30-min** natural gas bars.

Pipeline:
1. Load raw tick/bar CSV via `read_exported_df`
2. Resample to configurable bar size (default 30min)
3. `ContinuousIntradayPrep` for session-aware technical features
4. Forward-fill daily EIA storage + weather onto intraday bars
5. Build regime features from daily aggregation (causal)
6. Windowed dataset â†’ HybridMixtureNetwork with optional classification + VSN

In [ ]:
from dotenv import load_dotenv; load_dotenv('/workspace/MacrOS-Intel/MacrOSINT/dot.env')

import os
import sys
import copy
import math
import time as _time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

try:
    import optuna
    from optuna.pruners import MedianPruner
    from optuna.samplers import TPESampler
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    _OPTUNA = True
except ImportError:
    print('optuna not installed -- pip install optuna')
    _OPTUNA = False

warnings.filterwarnings('ignore')

print(f'PyTorch : {torch.__version__}')
print(f'Optuna  : {optuna.__version__ if _OPTUNA else "N/A"}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

In [ ]:
from CTAFlow.models.deep_learning.multi_branch.ng_moe import (
    HybridConfig,
    HybridMixtureNetwork,
    HybridLoss,
)
from CTAFlow.data.raw_formatting.intraday_manager import read_exported_df
from CTAFlow.models.prep.intraday_continuous import (
    ContinuousIntradayPrep,
    SessionSpec,
)

# macrOS-Int for EIA + weather
sys.path.insert(0, r'C:\Users\nicho\PycharmProjects\macrOS-Int')
from MacrOSINT.data.sources.eia.api_tools import NatGasHelper
from MacrOSINT.models.energy.natgas_storage_forecast import (
    NatGasStorageForecaster,
    fetch_storage_data,
    ConsensusForecast,
    compute_degree_days,
    compute_spline_hdd_basis,
)

# Device
def _select_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        t = torch.zeros(1, device='cuda')
        _ = t + 1
        return 'cuda'
    except RuntimeError as e:
        print(f'CUDA unusable ({e}) -- CPU fallback')
        return 'cpu'

DEVICE = _select_device()
print(f'Device: {DEVICE}')

## 1. Configuration

In [ ]:
# --- Paths ---
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    SAVE_DIR    = Path('/content/drive/MyDrive/results/ng_hybrid_intraday')
    DATA_DIR    = Path('/content/drive/MyDrive/features')
else:
    SAVE_DIR    = Path('/workspace/results/ng_hybrid_intraday')
    DATA_DIR    = Path('/workspace/model_data')

SAVE_DIR.mkdir(parents=True, exist_ok=True)

INTRADAY_CSV = DATA_DIR / 'NG' / 'intraday.csv'
EIA_HDF      = str(DATA_DIR / 'ng_eia_cache.hdf')
WEATHER_HDF  = str(DATA_DIR / 'weather.hdf')

# --- Intraday config ---
BAR_MINUTES       = 30          # resample frequency
SESSION           = SessionSpec('USA', '08:30', '15:00')  # configurable session
TARGET_HORIZON_BARS = 2         # forecast horizon in bars (2 * 30min = 60min)
SEQ_LEN           = 20          # lookback window in bars
AE_WINDOW_DAYS    = 21          # regime encoder lookback in trading days

# --- Classification config ---
N_CLASSES = 4                   # quartile-based: <25 / <50 / >50 / >75
USE_POSITIONING = True          # tanh exposure head
USE_VSN = True                  # grouped variable selection
MONDAY_ONLY = False             # sample every bar, not just Mondays

# --- Optuna ---
N_TRIALS       = 40
MAX_EPOCHS_OPT = 80
OPT_PATIENCE   = 10

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Save dir     : {SAVE_DIR}')
print(f'Intraday CSV : {INTRADAY_CSV}')
print(f'Bar freq     : {BAR_MINUTES}min')
print(f'Session      : {SESSION.name} ({SESSION.start}-{SESSION.end})')
print(f'Target       : {TARGET_HORIZON_BARS} bars = {TARGET_HORIZON_BARS * BAR_MINUTES}min')
print(f'Seq len      : {SEQ_LEN} bars')
print(f'Classes      : {N_CLASSES}')
print(f'Positioning  : {USE_POSITIONING}')
print(f'VSN          : {USE_VSN}')

## 2. Load & Resample Intraday Data

In [ ]:
# Load raw Sierra Chart CSV
raw_df = read_exported_df(str(INTRADAY_CSV))
print(f'Raw bars     : {raw_df.shape}')
print(f'Date range   : {raw_df.index[0]} to {raw_df.index[-1]}')
print(f'Columns      : {raw_df.columns.tolist()}')

# Standardize column names
raw_df.columns = [c.lower() for c in raw_df.columns]
for old, new in [('last', 'close'), ('vol', 'volume'), ('numberoftrades', 'ticks')]:
    if old in raw_df.columns and new not in raw_df.columns:
        raw_df.rename(columns={old: new}, inplace=True)

# Resample to BAR_MINUTES
resample_rule = f'{BAR_MINUTES}min'
ohlcv_agg = {
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
}
if 'volume' in raw_df.columns:
    ohlcv_agg['volume'] = 'sum'
if 'ticks' in raw_df.columns:
    ohlcv_agg['ticks'] = 'sum'

df = raw_df.resample(resample_rule).agg(ohlcv_agg).dropna(subset=['close'])

# Filter to session hours
session_start = pd.Timestamp(SESSION.start).time()
session_end = pd.Timestamp(SESSION.end).time()
df = df.between_time(session_start, session_end)

print(f'\nResampled    : {df.shape} @ {resample_rule}')
print(f'Session      : {SESSION.start}-{SESSION.end}')
print(f'Date range   : {df.index[0]} to {df.index[-1]}')
print(f'Bars/day     : ~{df.groupby(df.index.date).size().median():.0f}')
df.tail(5)

## 3. Load EIA Storage + Consensus

In [ ]:
START = df.index[0].strftime('%Y-%m')
END   = df.index[-1].strftime('%Y-%m')

eia_cache    = NatGasStorageForecaster.load_eia_cache(hdf_path=EIA_HDF)
storage_wkly = eia_cache.get('storage')

if storage_wkly is not None and not storage_wkly.empty:
    print(f'EIA storage from cache : {storage_wkly.shape}')
else:
    ng_helper    = NatGasHelper()
    storage_wkly = fetch_storage_data(ng_helper, start=START, end=END)
    NatGasStorageForecaster.save_eia_cache(storage=storage_wkly, hdf_path=EIA_HDF)
    print(f'EIA storage from API   : {storage_wkly.shape}')

# Consensus forecast for surprise features
try:
    cf = ConsensusForecast()
    cf.fit(storage_wkly['storage_change'])
    surprise_df = cf.transform()
    storage_wkly = storage_wkly.join(
        surprise_df[['consensus_est', 'surprise']], how='left'
    )
    print(f'ConsensusForecast added')
except Exception as e:
    print(f'ConsensusForecast failed ({e}) -- using rolling 4-week proxy')
    chg = storage_wkly['storage_change']
    storage_wkly['consensus_est'] = chg.rolling(4).mean()
    storage_wkly['surprise']      = chg - storage_wkly['consensus_est']

print(f'Weekly columns: {storage_wkly.columns.tolist()}')

## 4. Load Weather

HDD/CDD degree-day features forward-filled onto intraday bars.

In [ ]:
daily_weather = NatGasStorageForecaster.load_weather_hdf(hdf_path=WEATHER_HDF)

if daily_weather is not None and not daily_weather.empty:
    daily_weather = daily_weather[
        (daily_weather.index >= df.index[0].normalize()) &
        (daily_weather.index <= df.index[-1].normalize())
    ]
    print(f'Weather from cache : {daily_weather.shape}')
    print(f'Columns    : {daily_weather.columns.tolist()}')
else:
    from MacrOSINT.models.weather.population_weather import PopulationWeatherGrid
    print('No weather cache -- fetching via PopulationWeatherGrid...')
    forecaster = NatGasStorageForecaster()
    forecaster.setup()
    daily_weather = forecaster._fetch_weather_by_epoch(
        df.index[0].date(), df.index[-1].date()
    )
    NatGasStorageForecaster.save_weather_hdf(daily_weather, hdf_path=WEATHER_HDF)
    print(f'Weather fetched and cached: {daily_weather.shape}')

daily_weather.tail(3)

## 5. Intraday Feature Engineering

Build features using `ContinuousIntradayPrep` then overlay daily storage + weather.

In [ ]:
# --- ContinuousIntradayPrep for base technical features ---
prep = ContinuousIntradayPrep(
    sessions=[SESSION],
    bar_minutes=BAR_MINUTES,
)

bars_per_60m = 60 // BAR_MINUTES
steps_60m = TARGET_HORIZON_BARS  # in bars (already at target freq)

df_prep, train_mask, target_cols = prep.prepare(
    df.copy(),
    steps_60m=TARGET_HORIZON_BARS,
    keep_only_active=False,
    add_daily=True,
    add_overnight=True,
    add_deseas=True,
    add_time_features=True,
    add_resample_precalc=False,   # already at 30min, no sub-resample needed
    apply_scaling=False,
    add_bid_ask='bidvol' in df.columns or 'askvol' in df.columns,
    add_event_markers=False,
)

# ContinuousIntradayPrep._standardize_columns uppercases OHLCV (Close, Open, ...);
# downstream cells expect lowercase — normalize here.
df_prep.columns = [c.lower() for c in df_prep.columns]

tech_feature_cols = prep.get_feature_cols(
    steps_60m=TARGET_HORIZON_BARS,
    bar_minutes=BAR_MINUTES,
    add_resample_precalc=False,
    add_bid_ask=False,
    add_event_markers=False,
)
# Match lowered column names
tech_feature_cols = [c.lower() for c in tech_feature_cols]
tech_feature_cols = [c for c in tech_feature_cols if c in df_prep.columns]

print(f'Prepared bars: {df_prep.shape}')
print(f'Tech features: {len(tech_feature_cols)}')
print(f'Target cols  : {target_cols}')

In [ ]:
# === Forward-fill daily storage features onto intraday bars ===
# Use the same approach as NGMoEDataBuilder._add_storage_daily_features
sw = storage_wkly.copy()
sl = sw['storage_level']
sw['sl_4wk_mean'] = sl.rolling(4, min_periods=2).mean()
sw['sl_4wk_max'] = sl.rolling(4, min_periods=2).max()
sw['sl_4wk_min'] = sl.rolling(4, min_periods=2).min()
sw['sl_change_4wk_mean'] = sw['storage_change'].rolling(4, min_periods=2).mean()

storage_cols = ['storage_level', 'storage_change',
                'sl_4wk_mean', 'sl_4wk_max', 'sl_4wk_min', 'sl_change_4wk_mean']
for extra in ['consensus_est', 'surprise']:
    if extra in sw.columns:
        storage_cols.append(extra)

# Create daily index, forward-fill, then reindex to intraday
daily_idx = pd.date_range(sw.index[0], df_prep.index[-1].normalize(), freq='D')
storage_daily = sw[storage_cols].reindex(
    sw.index.union(daily_idx)
).sort_index().ffill()

# Map each intraday bar to its date, then lookup
bar_dates = df_prep.index.normalize()
for col in storage_cols:
    df_prep[col] = storage_daily[col].reindex(bar_dates).values

storage_feature_cols = list(storage_cols)
print(f'Storage features: {len(storage_feature_cols)} -> {storage_feature_cols}')

# === Forward-fill daily weather features onto intraday bars ===
weather_feature_cols = []
if daily_weather is not None and not daily_weather.empty:
    dd = compute_degree_days(daily_weather)

    # HDD/CDD daily + 7-day rolling
    dd['HDD_7d'] = dd['HDD'].rolling(7, min_periods=3).sum()
    dd['CDD_7d'] = dd['CDD'].rolling(7, min_periods=3).sum()
    dd['HDD_7d_chg'] = dd['HDD_7d'] - dd['HDD_7d'].shift(7)
    dd['CDD_7d_chg'] = dd['CDD_7d'] - dd['CDD_7d'].shift(7)

    dd_cols = ['HDD', 'CDD', 'HDD_7d', 'CDD_7d', 'HDD_7d_chg', 'CDD_7d_chg']

    # Spline HDD basis
    try:
        hdd_7d_series = dd['HDD_7d'].fillna(0)
        spline_df, _ = compute_spline_hdd_basis(hdd_7d_series, n_knots=4)
        for sc in spline_df.columns:
            dd[sc] = spline_df[sc].values
            dd_cols.append(sc)
    except Exception:
        pass

    # Population-weighted temperature
    if 'wtd_TAVG' in daily_weather.columns:
        dd['wtd_tavg'] = daily_weather['wtd_TAVG'].values
        dd['wtd_tavg_7d'] = daily_weather['wtd_TAVG'].rolling(7, min_periods=3).mean().values
        dd_cols.extend(['wtd_tavg', 'wtd_tavg_7d'])

    # Forward-fill to intraday
    for col in dd_cols:
        renamed = f'dd_{col.lower()}' if not col.startswith('dd_') else col
        df_prep[renamed] = dd[col].reindex(bar_dates).values
        weather_feature_cols.append(renamed)

    print(f'Weather features: {len(weather_feature_cols)} -> {weather_feature_cols}')
else:
    print('No weather data available')

In [ ]:
# === Build daily regime features (12-dim VAE input) ===
# Aggregate intraday bars to daily, then compute regime features

daily_close = df_prep.groupby(df_prep.index.date)['close'].last()
daily_close.index = pd.DatetimeIndex(daily_close.index)
daily_log_ret = np.log(daily_close / daily_close.shift(1))

REGIME_COLS = [
    'regime_ret_1d', 'regime_ret_5d', 'regime_ret_21d',
    'regime_rv_5d', 'regime_rv_21d',
    'regime_pct_in_5y_band', 'regime_dev_5y_zscore', 'regime_band_width_pct',
    'regime_fc_vs_seasonal_z', 'regime_chg_vs_seasonal_z',
    'regime_is_injection', 'regime_dev_x_season',
]

# Returns & vol context (5 features)
regime_daily = pd.DataFrame(index=daily_close.index)
regime_daily['regime_ret_1d'] = daily_log_ret
regime_daily['regime_ret_5d'] = daily_log_ret.rolling(5).sum()
regime_daily['regime_ret_21d'] = daily_log_ret.rolling(21).sum()
regime_daily['regime_rv_5d'] = np.sqrt((daily_log_ret**2).rolling(5).mean()) * np.sqrt(252)
regime_daily['regime_rv_21d'] = np.sqrt((daily_log_ret**2).rolling(21).mean()) * np.sqrt(252)

# Storage state (3 features) -- 5-year band per ISO week (causal)
sl = storage_wkly['storage_level']
wk_idx = sl.index.isocalendar().week.values
hi_5y = pd.Series(np.nan, index=sl.index)
lo_5y = pd.Series(np.nan, index=sl.index)
mean_5y = pd.Series(np.nan, index=sl.index)

for w in range(1, 54):
    mask = wk_idx == w
    if mask.sum() < 2:
        continue
    idx_pos = np.where(mask)[0]
    vals = sl.iloc[idx_pos]
    hi_5y.iloc[idx_pos] = vals.expanding().max().shift(1).values
    lo_5y.iloc[idx_pos] = vals.expanding().min().shift(1).values
    mean_5y.iloc[idx_pos] = vals.expanding().mean().shift(1).values

hi_5y = hi_5y.ffill().bfill()
lo_5y = lo_5y.ffill().bfill()
mean_5y = mean_5y.ffill().bfill()

band_w = (hi_5y - lo_5y).clip(lower=1)
pct_band = (sl - lo_5y) / band_w
dev_zscore = (sl - mean_5y) / band_w

regime_wkly = pd.DataFrame({
    'regime_pct_in_5y_band': pct_band.values,
    'regime_dev_5y_zscore': dev_zscore.values,
    'regime_band_width_pct': (band_w / mean_5y.clip(lower=1)).values,
}, index=sl.index)
regime_wkly_daily = regime_wkly.reindex(
    regime_wkly.index.union(daily_close.index)
).sort_index().ffill().reindex(daily_close.index)
for col in regime_wkly_daily.columns:
    regime_daily[col] = regime_wkly_daily[col].values

# Forecast vs seasonal (2 features)
sc = storage_wkly['storage_change']
sea_chg = pd.Series(np.nan, index=sc.index)
for w in range(1, 54):
    mask = wk_idx == w
    if mask.sum() < 2:
        continue
    idx_pos = np.where(mask)[0]
    sea_chg.iloc[idx_pos] = sc.iloc[idx_pos].expanding().mean().shift(1).values
sea_chg = sea_chg.ffill().bfill()
sea_std = (sc - sea_chg).expanding().std().clip(lower=1)
chg_vs_sea = (sc - sea_chg) / sea_std

if 'consensus_est' in storage_wkly.columns:
    fc = storage_wkly['consensus_est']
else:
    fc = sea_chg
fc_vs_sea = (fc - sea_chg) / sea_std

fc_regime = pd.DataFrame({
    'regime_fc_vs_seasonal_z': fc_vs_sea.values,
    'regime_chg_vs_seasonal_z': chg_vs_sea.values,
}, index=sc.index)
fc_daily = fc_regime.reindex(
    fc_regime.index.union(daily_close.index)
).sort_index().ffill().reindex(daily_close.index)
for col in fc_daily.columns:
    regime_daily[col] = fc_daily[col].values

# Seasonal flags (2 features)
month = daily_close.index.month
regime_daily['regime_is_injection'] = ((month >= 4) & (month <= 10)).astype(np.float32)
season_sign = np.where(regime_daily['regime_is_injection'].values > 0.5, 1.0, -1.0)
regime_daily['regime_dev_x_season'] = regime_daily['regime_dev_5y_zscore'] * season_sign

# Forward-fill regime to intraday
for col in REGIME_COLS:
    df_prep[col] = regime_daily[col].reindex(bar_dates).values

regime_daily = regime_daily.dropna()
print(f'Regime daily rows: {len(regime_daily)}')
print(f'Regime cols: {REGIME_COLS}')

## 6. Build Windowed Dataset

Intraday bars â†’ sliding window samples with (x_seq, ae_input, y_ret, y_std, y_class).

In [ ]:
# --- Assemble all feature columns in order, track groups ---
feature_groups = {}
all_feature_cols = []

# Technical group from ContinuousIntradayPrep
feature_groups['technical'] = list(tech_feature_cols)
all_feature_cols.extend(tech_feature_cols)

# Storage group
feature_groups['storage'] = list(storage_feature_cols)
all_feature_cols.extend(storage_feature_cols)

# Weather group
if weather_feature_cols:
    feature_groups['weather'] = list(weather_feature_cols)
    all_feature_cols.extend(weather_feature_cols)

FEATURE_GROUP_SIZES = {k: len(v) for k, v in feature_groups.items()}
n_features = len(all_feature_cols)

print(f'Total features: {n_features}')
print(f'Feature groups ({len(FEATURE_GROUP_SIZES)}):')
for gname, gsize in FEATURE_GROUP_SIZES.items():
    print(f'  {gname:15s}: {gsize:2d} features')

# --- Compute target: forward log return over TARGET_HORIZON_BARS ---
target_col = f'y_fwd_{TARGET_HORIZON_BARS}'
if target_col not in df_prep.columns:
    df_prep[target_col] = np.log(
        df_prep['close'].shift(-TARGET_HORIZON_BARS) / df_prep['close']
    )
df_prep['target'] = df_prep[target_col]

# Target std: rolling realized vol of bar returns
bar_ret = np.log(df_prep['close'] / df_prep['close'].shift(1))
bars_per_day = int(df_prep.groupby(df_prep.index.date).size().median())
df_prep['target_std'] = bar_ret.rolling(
    bars_per_day * 5, min_periods=bars_per_day
).std() * np.sqrt(TARGET_HORIZON_BARS)

# --- Classification target (expanding quantile, causal) ---
if N_CLASSES > 0:
    from CTAFlow.models.deep_learning.multi_branch.ng_moe_dataset import (
        NGMoEDataBuilder,
    )
    df_prep['target_class'] = NGMoEDataBuilder._compute_target_classes(
        df_prep['target'], N_CLASSES, None,
    )

# --- Drop warm-up NaNs ---
keep_cols = all_feature_cols + REGIME_COLS + ['target', 'target_std']
if 'target_class' in df_prep.columns:
    keep_cols.append('target_class')
keep_cols = [c for c in keep_cols if c in df_prep.columns]
df_clean = df_prep.dropna(subset=keep_cols).copy()
print(f'Clean bars: {len(df_clean)} (dropped {len(df_prep) - len(df_clean)} warm-up/nan)')

In [ ]:
from CTAFlow.data.datasets.intraday_hybrid import IntradayHybridDataset

# --- Chronological split ---
n_total = len(df_clean)
n_tr = int(n_total * 0.70)
n_va = int(n_total * 0.15)

train_df = df_clean.iloc[:n_tr]
val_df   = df_clean.iloc[n_tr:n_tr + n_va]
test_df  = df_clean.iloc[n_tr + n_va:]

ds_kw = dict(
    feature_cols=all_feature_cols,
    regime_cols=REGIME_COLS,
    seq_len=SEQ_LEN,
    ae_window_days=AE_WINDOW_DAYS,
    bars_per_day=bars_per_day,
    n_classes=N_CLASSES,
)
train_ds = IntradayHybridDataset(train_df, stride=TARGET_HORIZON_BARS, **ds_kw)
val_ds   = IntradayHybridDataset(val_df, stride=1, **ds_kw)
test_ds  = IntradayHybridDataset(test_df, stride=1, **ds_kw)

ae_window_bars = AE_WINDOW_DAYS * bars_per_day

print(f'Bars/day     : {bars_per_day}')
print(f'AE window    : {AE_WINDOW_DAYS}d = {ae_window_bars} bars')
print(f'Dataset stride: train={TARGET_HORIZON_BARS}, val/test=1')
print(f'Train samples: {len(train_ds)}  ({train_df.index[0].date()} - {train_df.index[-1].date()})')
print(f'Val samples  : {len(val_ds)}  ({val_df.index[0].date()} - {val_df.index[-1].date()})')
print(f'Test samples : {len(test_ds)}  ({test_df.index[0].date()} - {test_df.index[-1].date()})')

# Verify sample shapes
sample = train_ds[0]
print(f'\nSample shapes: {[tuple(s.shape) if hasattr(s, "shape") else s.item() for s in sample]}')

# Class distribution
if train_ds.has_classes:
    train_classes = train_ds.y_class[train_ds.indices]
    for c in range(N_CLASSES):
        pct = (train_classes == c).mean() * 100
        print(f'  Class {c}: {pct:.1f}%')

## 7. Selection Score & Metrics

In [ ]:
def compute_val_metrics(model, val_loader, loss_fn, device):
    """Evaluate model on validation set, return comprehensive metrics dict."""
    model.eval()
    all_losses = {k: [] for k in ['total_loss', 'return_loss', 'vol_loss',
                                    'mdn_nll_loss', 'ce_loss', 'positioning_loss',
                                    'vsn_entropy_loss']}
    all_positions, all_returns, all_pred_returns = [], [], []
    all_pred_classes, all_true_classes, all_vsn_weights = [], [], []
    n_total = 0

    with torch.no_grad():
        for batch in val_loader:
            if len(batch) == 5:
                x_seq, ae_in, y_ret, y_std, y_cls = batch
                y_cls = y_cls.to(device)
            else:
                x_seq, ae_in, y_ret, y_std = batch
                y_cls = None

            x_seq = x_seq.to(device); ae_in = ae_in.to(device)
            y_ret = y_ret.to(device); y_std = y_std.to(device)

            out = model(x_seq, ae_in)
            losses = loss_fn(out, y_ret, y_std, y_cls)
            for k in all_losses:
                if k in losses:
                    v = losses[k]
                    all_losses[k].append((v.item() if torch.is_tensor(v) else v) * len(y_ret))
            n_total += len(y_ret)
            all_pred_returns.append(out['pred_return'].cpu().numpy())
            all_returns.append(y_ret.cpu().numpy())
            if 'position' in out:
                all_positions.append(out['position'].cpu().numpy())
            if 'class_logits' in out and y_cls is not None:
                all_pred_classes.append(out['class_logits'].argmax(dim=-1).cpu().numpy())
                all_true_classes.append(y_cls.cpu().numpy())
            if 'vsn_weights' in out:
                all_vsn_weights.append(out['vsn_weights'].cpu().numpy())

    metrics = {}
    for k, vals in all_losses.items():
        if vals:
            metrics[k] = sum(vals) / max(n_total, 1)
    metrics['val_loss'] = metrics.get('total_loss', 0)

    pred_ret = np.concatenate(all_pred_returns)
    actual_ret = np.concatenate(all_returns)
    metrics['return_mae'] = np.abs(pred_ret - actual_ret).mean()
    metrics['return_corr'] = np.corrcoef(pred_ret, actual_ret)[0, 1] if len(pred_ret) > 2 else 0
    dir_mask = np.abs(actual_ret) > 1e-6
    metrics['direction_accuracy'] = (np.sign(pred_ret[dir_mask]) == np.sign(actual_ret[dir_mask])).mean() if dir_mask.any() else 0

    if all_pred_classes and all_true_classes:
        pc, ac = np.concatenate(all_pred_classes), np.concatenate(all_true_classes)
        metrics['accuracy'] = (pc == ac).mean()
        for c in range(int(ac.max()) + 1):
            m = ac == c
            if m.sum() > 0:
                metrics[f'acc_class_{c}'] = (pc[m] == c).mean()
                metrics[f'n_class_{c}'] = int(m.sum())

    if all_positions:
        pos = np.concatenate(all_positions)
        sr = pos * actual_ret
        cum = np.cumsum(sr)
        std = sr.std()
        metrics['sharpe'] = (sr.mean() / std * np.sqrt(252 * bars_per_day)) if std > 0 else 0
        ds = sr[sr < 0]
        metrics['sortino'] = (sr.mean() / ds.std() * np.sqrt(252 * bars_per_day)) if len(ds) > 1 and ds.std() > 0 else 0
        g, l = sr[sr > 0].sum(), abs(sr[sr < 0].sum())
        metrics['profit_factor'] = g / max(l, 1e-8)
        metrics['win_rate'] = (sr > 0).mean()
        metrics['mean_abs_position'] = np.abs(pos).mean()
        metrics['position_std'] = pos.std()
        running_max = np.maximum.accumulate(cum)
        metrics['max_drawdown'] = (cum - running_max).min()
        metrics['mean_strategy_ret'] = sr.mean()

    if all_vsn_weights:
        vsn_w = np.concatenate(all_vsn_weights)
        for i, name in enumerate(FEATURE_GROUP_SIZES.keys()):
            metrics[f'vsn_{name}'] = vsn_w[:, i].mean()

    return metrics


def selection_score(metrics):
    if USE_POSITIONING:
        sharpe = metrics.get('sharpe', 0)
        sortino = metrics.get('sortino', 0)
        pf = metrics.get('profit_factor', 1e-8)
        acc = metrics.get('accuracy', 0.25)
        return 0.25*sharpe + 0.30*sortino + 0.20*math.log(max(pf, 1e-8)) + 0.25*(acc-0.25)*10
    else:
        return -metrics['val_loss'] + 5.0 * (metrics.get('accuracy', 0.25) - 0.25)


def _fmt_metrics(metrics, prefix=''):
    lines = []
    loss_keys = ['val_loss', 'return_loss', 'vol_loss', 'mdn_nll_loss', 'ce_loss', 'positioning_loss']
    lp = [f'{k}={metrics[k]:.5f}' for k in loss_keys if k in metrics]
    if lp: lines.append(f'{prefix}Losses   : {", ".join(lp)}')
    if 'accuracy' in metrics:
        cp = [f'acc={metrics["accuracy"]:.3f}']
        for c in range(10):
            if f'acc_class_{c}' in metrics:
                cp.append(f'c{c}={metrics[f"acc_class_{c}"]:.3f}({metrics.get(f"n_class_{c}",0)})')
        lines.append(f'{prefix}Class    : {", ".join(cp)}')
    if 'sharpe' in metrics:
        pp = [f'sharpe={metrics["sharpe"]:.3f}', f'sortino={metrics["sortino"]:.3f}',
              f'pf={metrics.get("profit_factor",0):.2f}', f'win={metrics.get("win_rate",0):.1%}',
              f'maxDD={metrics.get("max_drawdown",0):.4f}']
        lines.append(f'{prefix}Position : {", ".join(pp)}')
    rp = [f'mae={metrics.get("return_mae",0):.5f}', f'corr={metrics.get("return_corr",0):.3f}',
          f'dir_acc={metrics.get("direction_accuracy",0):.3f}']
    lines.append(f'{prefix}Returns  : {", ".join(rp)}')
    vp = [f'{k.replace("vsn_","")}={v:.3f}' for k,v in metrics.items() if k.startswith('vsn_')]
    if vp: lines.append(f'{prefix}VSN      : {", ".join(vp)}')
    return '\n'.join(lines)

print('Metrics + selection score defined')

## 8. Optuna Objective

In [ ]:
_trial_log = []

def _make_loaders(train_ds, val_ds, bs):
    return (DataLoader(train_ds, batch_size=bs, shuffle=True, drop_last=True),
            DataLoader(val_ds, batch_size=bs, shuffle=False, drop_last=False))

def objective(trial):
    t0 = _time.time()

    d_latent    = trial.suggest_categorical('d_latent', [16, 32, 64])
    d_ae_hidden = trial.suggest_categorical('d_ae_hidden', [64, 128, 256])
    tcn_width   = trial.suggest_categorical('tcn_width', [32, 64, 128])
    tcn_depth   = trial.suggest_int('tcn_depth', 2, 4)
    stride      = trial.suggest_categorical('stride', [1, 2, 3, 4])
    mdn_hidden  = trial.suggest_categorical('mdn_hidden', [32, 64, 128])
    mdn_n_comp  = trial.suggest_int('mdn_n_components', 2, 6)
    head_hidden = trial.suggest_categorical('head_hidden_dim', [64, 128, 256])
    dropout     = trial.suggest_float('dropout', 0.05, 0.4)
    pos_hidden  = trial.suggest_categorical('positioning_hidden_dim', [16, 32, 64])

    if USE_VSN:
        vsn_d_model        = trial.suggest_categorical('vsn_d_model', [16, 32, 64])
        vsn_temperature    = trial.suggest_float('vsn_temperature', 0.5, 3.0)
        vsn_entropy_weight = trial.suggest_float('vsn_entropy_weight', 0.01, 0.3, log=True)
        vsn_min_weight     = trial.suggest_float('vsn_min_weight', 0.0, 0.1)
    else:
        vsn_d_model, vsn_temperature, vsn_entropy_weight, vsn_min_weight = 32, 1.5, 0.1, 0.05

    ce_weight    = trial.suggest_float('ce_weight', 0.3, 3.0, log=True)
    pnl_weight   = trial.suggest_float('positioning_pnl_weight', 0.1, 2.0, log=True)
    nll_weight   = trial.suggest_float('nll_weight', 0.01, 0.5, log=True)
    kl_weight    = trial.suggest_float('kl_weight', 0.001, 0.1, log=True)
    recon_weight = trial.suggest_float('recon_weight', 0.01, 0.5, log=True)
    tc_cost      = trial.suggest_float('tc_cost', 0.0, 0.001)
    lr           = trial.suggest_float('lr', 1e-4, 3e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)
    batch_size   = trial.suggest_categorical('batch_size', [32, 64, 128])

    config = HybridConfig(
        n_features=n_features, seq_len=SEQ_LEN,
        f_ae=12, ae_window=ae_window_bars, d_latent=d_latent, d_ae_hidden=d_ae_hidden,
        kl_weight=kl_weight, recon_weight=recon_weight,
        tcn_channels=[tcn_width]*tcn_depth, tcn_kernel_size=3, stride=stride,
        mdn_hidden_dims=[mdn_hidden, mdn_hidden//2], mdn_n_components=mdn_n_comp,
        head_hidden_dim=head_hidden, dropout=dropout, nll_weight=nll_weight,
        n_classes=N_CLASSES, use_positioning_head=USE_POSITIONING,
        positioning_hidden_dim=pos_hidden,
        ce_weight=ce_weight, positioning_pnl_weight=pnl_weight, tc_cost=tc_cost,
        use_vsn=USE_VSN, vsn_d_model=vsn_d_model, vsn_temperature=vsn_temperature,
        vsn_entropy_weight=vsn_entropy_weight, vsn_min_weight=vsn_min_weight,
    )

    model = HybridMixtureNetwork(
        config, feature_group_sizes=FEATURE_GROUP_SIZES if USE_VSN else None,
    ).to(DEVICE)
    loss_fn = HybridLoss(config)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)
    train_loader, val_loader = _make_loaders(train_ds, val_ds, batch_size)

    n_params = sum(p.numel() for p in model.parameters())
    best_score, best_metrics, patience_cnt = float('-inf'), {}, 0

    # Current study best for comparison
    try:
        study_best = study.best_value
    except ValueError:
        study_best = float('-inf')

    for epoch in range(1, MAX_EPOCHS_OPT + 1):
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        for batch in train_loader:
            if len(batch) == 5:
                x_seq, ae_in, y_ret, y_std, y_cls = batch
                y_cls = y_cls.to(DEVICE)
            else:
                x_seq, ae_in, y_ret, y_std = batch; y_cls = None
            x_seq, ae_in = x_seq.to(DEVICE), ae_in.to(DEVICE)
            y_ret, y_std = y_ret.to(DEVICE), y_std.to(DEVICE)
            optimizer.zero_grad()
            out = model(x_seq, ae_in)
            losses = loss_fn(out, y_ret, y_std, y_cls)
            losses['total_loss'].backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += losses['total_loss'].item()
            n_batches += 1
        scheduler.step()

        metrics = compute_val_metrics(model, val_loader, loss_fn, DEVICE)
        score = selection_score(metrics)
        trial.report(score, epoch)

        # --- Every 5 epochs: print progress ---
        if epoch % 5 == 0 or epoch == 1:
            elapsed = _time.time() - t0
            avg_train = epoch_loss / max(n_batches, 1)
            print(f'  T{trial.number:02d} E{epoch:3d} | '
                  f'train_loss={avg_train:.4f} val_loss={metrics["val_loss"]:.4f} '
                  f'acc={metrics.get("accuracy",0):.3f} '
                  f'sharpe={metrics.get("sharpe",0):.3f} '
                  f'score={score:.3f} best={best_score:.3f} '
                  f'study_best={study_best:.3f} ({elapsed:.0f}s)')

        if trial.should_prune():
            _trial_log.append({'trial': trial.number, 'status': 'PRUNED',
                               'epoch': epoch, 'score': score, 'val_loss': metrics['val_loss'],
                               'time': _time.time()-t0})
            raise optuna.TrialPruned()
        if score > best_score:
            best_score, best_metrics, patience_cnt = score, metrics.copy(), 0
        else:
            patience_cnt += 1
            if patience_cnt >= OPT_PATIENCE:
                break

    elapsed = _time.time() - t0
    print(f'\n{"="*70}')
    print(f'Trial {trial.number:3d} COMPLETE  epochs={epoch}  score={best_score:.3f}  '
          f'params={n_params:,}  ({elapsed:.0f}s)')
    print(f'  Arch: tcn=[{tcn_width}]*{tcn_depth} stride={stride} mdn_h={mdn_hidden} head={head_hidden} '
          f'd_latent={d_latent} vsn_d={vsn_d_model if USE_VSN else "off"}')
    print(_fmt_metrics(best_metrics, prefix='  '))
    print(f'{"="*70}')

    log_entry = {'trial': trial.number, 'status': 'COMPLETE', 'epoch': epoch,
                 'score': best_score, 'n_params': n_params, 'time': elapsed}
    for k in ['val_loss','accuracy','sharpe','sortino','profit_factor','win_rate',
              'max_drawdown','direction_accuracy','return_corr','mean_abs_position']:
        log_entry[k] = best_metrics.get(k, 0)
    for c in range(N_CLASSES):
        log_entry[f'acc_class_{c}'] = best_metrics.get(f'acc_class_{c}', 0)
    for name in FEATURE_GROUP_SIZES:
        log_entry[f'vsn_{name}'] = best_metrics.get(f'vsn_{name}', 0)
    _trial_log.append(log_entry)
    return best_score

print(f'Objective defined (n_features={n_features}, ae_window={ae_window_bars} bars)')

## 9. Run Optuna

In [ ]:
STUDY_NAME = f'ng_hybrid_intraday_{BAR_MINUTES}min_cls{N_CLASSES}'

study = optuna.create_study(
    study_name=STUDY_NAME, direction='maximize',
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=5),
)
print(f'Study    : {STUDY_NAME}')
print(f'Trials   : {N_TRIALS}')
print(f'Bar freq : {BAR_MINUTES}min  |  Horizon: {TARGET_HORIZON_BARS} bars')
print('-' * 60)

In [ ]:
# --- Select trial (default: best; set TRIAL_RANK to use 2nd-best, 3rd-best, etc.) ---
TRIAL_RANK = 0  # 0 = best, 1 = 2nd-best, 2 = 3rd-best, ...

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

sorted_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else float('-inf'), reverse=True)
TRIAL_RANK = min(TRIAL_RANK, len(sorted_trials) - 1)
selected_trial = sorted_trials[TRIAL_RANK]

print(f'\nTop-5 trials:')
for i, t in enumerate(sorted_trials[:5]):
    marker = ' <-- selected' if i == TRIAL_RANK else ''
    print(f'  #{t.number}: {t.value:.4f}{marker}')

print(f'\nUsing trial #{selected_trial.number} (rank {TRIAL_RANK + 1})')
print(f'Score: {selected_trial.value:.4f}')
for k, v in selected_trial.params.items():
    print(f'  {k}: {v}')

In [ ]:
import json

best_params = dict(selected_trial.params)
best_params['best_value'] = selected_trial.value
best_params['trial_number'] = selected_trial.number
best_params['trial_rank'] = TRIAL_RANK
best_params['n_classes'] = N_CLASSES
best_params['bar_minutes'] = BAR_MINUTES
best_params['target_horizon_bars'] = TARGET_HORIZON_BARS
best_params['session'] = f'{SESSION.name}_{SESSION.start}-{SESSION.end}'
best_params['n_features'] = n_features
best_params['ae_window_bars'] = ae_window_bars

with open(SAVE_DIR / 'best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)

trial_df = pd.DataFrame(_trial_log)
trial_df.to_csv(SAVE_DIR / 'trial_log.csv', index=False)
print(f'Saved to {SAVE_DIR}')
trial_df.sort_values('score', ascending=False).head(10)

## 10. Optuna Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
comp = trial_df[trial_df['status'] == 'COMPLETE'].copy()
best_so_far = comp['score'].expanding().max()
axes[0, 0].scatter(comp['trial'], comp['score'], s=20, alpha=0.6, label='trial score')
axes[0, 0].plot(comp['trial'].values, best_so_far.values, color='red', lw=1.5, label='best so far')
axes[0, 0].set_xlabel('Trial'); axes[0, 0].set_ylabel('Score')
axes[0, 0].set_title('Optimization History'); axes[0, 0].legend(fontsize=8)

try:
    imp = optuna.importance.get_param_importances(study)
    names, vals = list(imp.keys())[:12], list(imp.values())[:12]
    axes[0, 1].barh(names, vals, color='steelblue', edgecolor='black')
    axes[0, 1].set_xlabel('Importance'); axes[0, 1].set_title('Param Importance')
except Exception:
    axes[0, 1].text(0.5, 0.5, 'Not enough data', ha='center', va='center',
                    transform=axes[0, 1].transAxes)

if 'accuracy' in comp.columns:
    axes[1, 0].scatter(comp['accuracy'], comp['score'], s=20, alpha=0.6, c='green')
    axes[1, 0].set_xlabel('Val Accuracy'); axes[1, 0].set_ylabel('Score')
    axes[1, 0].set_title('Score vs Accuracy')

if 'sharpe' in comp.columns:
    sc = axes[1, 1].scatter(comp['sharpe'], comp['sortino'], s=20, alpha=0.6,
                            c=comp['score'], cmap='viridis')
    axes[1, 1].set_xlabel('Sharpe'); axes[1, 1].set_ylabel('Sortino')
    axes[1, 1].set_title('Sharpe vs Sortino'); plt.colorbar(sc, ax=axes[1, 1], label='Score')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'optuna_viz.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Train Final Model

In [ ]:
bp = selected_trial.params

final_cfg = HybridConfig(
    n_features=n_features, seq_len=SEQ_LEN,
    f_ae=12, ae_window=ae_window_bars,
    d_latent=bp['d_latent'], d_ae_hidden=bp['d_ae_hidden'],
    kl_weight=bp['kl_weight'], recon_weight=bp['recon_weight'],
    tcn_channels=[bp['tcn_width']]*bp['tcn_depth'], tcn_kernel_size=3,
    stride=bp['stride'],
    mdn_hidden_dims=[bp['mdn_hidden'], bp['mdn_hidden']//2],
    mdn_n_components=bp['mdn_n_components'],
    head_hidden_dim=bp['head_hidden_dim'], dropout=bp['dropout'],
    nll_weight=bp['nll_weight'],
    n_classes=N_CLASSES, use_positioning_head=USE_POSITIONING,
    positioning_hidden_dim=bp['positioning_hidden_dim'],
    ce_weight=bp['ce_weight'], positioning_pnl_weight=bp['positioning_pnl_weight'],
    tc_cost=bp['tc_cost'],
    use_vsn=USE_VSN,
    vsn_d_model=bp.get('vsn_d_model', 32),
    vsn_temperature=bp.get('vsn_temperature', 1.5),
    vsn_entropy_weight=bp.get('vsn_entropy_weight', 0.1),
    vsn_min_weight=bp.get('vsn_min_weight', 0.05),
)

model = HybridMixtureNetwork(
    final_cfg, feature_group_sizes=FEATURE_GROUP_SIZES if USE_VSN else None,
).to(DEVICE)
loss_fn = HybridLoss(final_cfg)
optimizer = optim.AdamW(model.parameters(), lr=bp['lr'], weight_decay=bp['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2)

FINAL_BS = bp['batch_size']
train_loader = DataLoader(train_ds, batch_size=FINAL_BS, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=FINAL_BS, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=FINAL_BS, shuffle=False)

print(f'Model params : {sum(p.numel() for p in model.parameters()):,}')
print(f'Trial        : #{selected_trial.number} (rank {TRIAL_RANK + 1})')
print(f'Stride       : {bp["stride"]}')
print(f'VSN groups   : {list(FEATURE_GROUP_SIZES.keys()) if USE_VSN else "disabled"}')

In [ ]:
MAX_EPOCHS = 200
PATIENCE   = 20
LOG_EVERY  = 20

loss_keys = ['total_loss', 'return_loss', 'vol_loss', 'mdn_nll_loss',
             'ae_recon_loss', 'ae_kl_loss', 'vsn_entropy_loss']
if N_CLASSES > 0: loss_keys.extend(['ce_loss', 'class_accuracy'])
if USE_POSITIONING: loss_keys.extend(['positioning_loss', 'mean_position', 'mean_strategy_ret'])

history = {f'train_{k}': [] for k in loss_keys}
history.update({f'val_{k}': [] for k in loss_keys})
for k in ['val_sharpe','val_sortino','val_score','val_profit_factor',
          'val_win_rate','val_max_drawdown','val_direction_accuracy','val_return_corr']:
    history[k] = []
for gname in FEATURE_GROUP_SIZES:
    history[f'val_vsn_{gname}'] = []

best_val_loss, best_state, best_metrics, patience_cnt = float('inf'), None, {}, 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_train = {k: [] for k in loss_keys}
    for batch in train_loader:
        if len(batch) == 5:
            x_seq, ae_in, y_ret, y_std, y_cls = batch; y_cls = y_cls.to(DEVICE)
        else:
            x_seq, ae_in, y_ret, y_std = batch; y_cls = None
        x_seq, ae_in = x_seq.to(DEVICE), ae_in.to(DEVICE)
        y_ret, y_std = y_ret.to(DEVICE), y_std.to(DEVICE)
        optimizer.zero_grad()
        out = model(x_seq, ae_in)
        losses = loss_fn(out, y_ret, y_std, y_cls)
        losses['total_loss'].backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        for k in loss_keys:
            if k in losses:
                v = losses[k]
                epoch_train[k].append(v.item() if torch.is_tensor(v) else v)
    scheduler.step()

    model.eval()
    epoch_val = {k: [] for k in loss_keys}
    with torch.no_grad():
        for batch in val_loader:
            if len(batch) == 5:
                x_seq, ae_in, y_ret, y_std, y_cls = batch; y_cls = y_cls.to(DEVICE)
            else:
                x_seq, ae_in, y_ret, y_std = batch; y_cls = None
            x_seq, ae_in = x_seq.to(DEVICE), ae_in.to(DEVICE)
            y_ret, y_std = y_ret.to(DEVICE), y_std.to(DEVICE)
            out = model(x_seq, ae_in)
            losses = loss_fn(out, y_ret, y_std, y_cls)
            for k in loss_keys:
                if k in losses:
                    v = losses[k]
                    epoch_val[k].append(v.item() if torch.is_tensor(v) else v)

    for k in loss_keys:
        history[f'train_{k}'].append(np.mean(epoch_train[k]) if epoch_train[k] else 0)
        history[f'val_{k}'].append(np.mean(epoch_val[k]) if epoch_val[k] else 0)

    val_loss = history['val_total_loss'][-1]
    is_new_best = val_loss < best_val_loss
    if is_new_best:
        best_val_loss = val_loss; best_state = copy.deepcopy(model.state_dict()); patience_cnt = 0
    else:
        patience_cnt += 1

    should_log = (epoch % LOG_EVERY == 0) or (epoch == 1) or is_new_best
    if should_log:
        val_metrics = compute_val_metrics(model, val_loader, loss_fn, DEVICE)
        score = selection_score(val_metrics)
        if is_new_best: best_metrics = val_metrics.copy()
    else:
        val_metrics = {}
        score = history['val_score'][-1] if history['val_score'] else 0

    def _get(key, dk=None):
        if key in val_metrics: return val_metrics[key]
        h = history.get(dk or f'val_{key}', []); return h[-1] if h else 0

    history['val_sharpe'].append(_get('sharpe','val_sharpe'))
    history['val_sortino'].append(_get('sortino','val_sortino'))
    history['val_score'].append(score)
    history['val_profit_factor'].append(_get('profit_factor','val_profit_factor'))
    history['val_win_rate'].append(_get('win_rate','val_win_rate'))
    history['val_max_drawdown'].append(_get('max_drawdown','val_max_drawdown'))
    history['val_direction_accuracy'].append(_get('direction_accuracy','val_direction_accuracy'))
    history['val_return_corr'].append(_get('return_corr','val_return_corr'))
    for gn in FEATURE_GROUP_SIZES:
        history[f'val_vsn_{gn}'].append(_get(f'vsn_{gn}', f'val_vsn_{gn}'))

    if should_log:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'\n--- Epoch {epoch:3d}/{MAX_EPOCHS} {"[NEW BEST]" if is_new_best else ""} '
              f'(patience={patience_cnt}/{PATIENCE}, lr={lr_now:.2e}) ---')
        print(f'  Train: total={history["train_total_loss"][-1]:.5f}  '
              f'nll={history["train_mdn_nll_loss"][-1]:.5f}')
        print(f'  Val  : total={val_loss:.5f}  nll={history["val_mdn_nll_loss"][-1]:.5f}')
        print(_fmt_metrics(val_metrics, prefix='  '))
        print(f'  Score={score:.3f}')

    if patience_cnt >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}'); break

print(f'\n{"="*70}')
print(f'Training complete  epochs={epoch}  best_val_loss={best_val_loss:.5f}')
print(_fmt_metrics(best_metrics, prefix='  '))
print(f'  Score={selection_score(best_metrics):.3f}')
print(f'{"="*70}')

In [ ]:
if best_state:
    model.load_state_dict(best_state)
    print('Loaded best model state')

## 12. Training Diagnostics

In [ ]:
ep = range(1, len(history['train_total_loss']) + 1)
fig, axes = plt.subplots(3, 4, figsize=(20, 12))

axes[0,0].plot(ep, history['train_total_loss'], label='Train')
axes[0,0].plot(ep, history['val_total_loss'], label='Val')
axes[0,0].set_title('Total Loss'); axes[0,0].legend()

axes[0,1].plot(ep, history['train_return_loss'], label='Train')
axes[0,1].plot(ep, history['val_return_loss'], label='Val')
axes[0,1].set_title('Return Loss'); axes[0,1].legend()

axes[0,2].plot(ep, history['train_vol_loss'], label='Train')
axes[0,2].plot(ep, history['val_vol_loss'], label='Val')
axes[0,2].set_title('Vol Loss'); axes[0,2].legend()

axes[0,3].plot(ep, history['val_mdn_nll_loss'], color='darkorange')
axes[0,3].set_title('MDN NLL (Val)')

if 'val_class_accuracy' in history and history['val_class_accuracy']:
    axes[1,0].plot(ep, history['train_class_accuracy'], label='Train')
    axes[1,0].plot(ep, history['val_class_accuracy'], label='Val')
    axes[1,0].axhline(1/N_CLASSES, color='red', ls='--', label='Chance')
    axes[1,0].set_title(f'{N_CLASSES}-Class Accuracy'); axes[1,0].legend()
else:
    axes[1,0].axis('off')

axes[1,1].plot(ep, history['val_ae_recon_loss'], label='Recon')
axes[1,1].plot(ep, history['val_ae_kl_loss'], label='KL')
axes[1,1].set_title('AE Losses'); axes[1,1].legend()

if 'val_positioning_loss' in history:
    axes[1,2].plot(ep, history['val_positioning_loss'], color='purple')
    axes[1,2].set_title('Positioning Loss')
else: axes[1,2].axis('off')

if 'val_mean_strategy_ret' in history:
    axes[1,3].plot(ep, np.cumsum(history['val_mean_strategy_ret']), color='green')
    axes[1,3].axhline(0, color='black', lw=0.5); axes[1,3].set_title('Cum Val Return')
else: axes[1,3].axis('off')

axes[2,0].plot(ep, history['val_sharpe'], color='teal'); axes[2,0].set_title('Sharpe')
axes[2,1].plot(ep, history['val_sortino'], color='darkorange'); axes[2,1].set_title('Sortino')
axes[2,2].plot(ep, history['val_score'], color='crimson'); axes[2,2].set_title('Score')
axes[2,3].plot(ep, history.get('val_ce_loss', []), color='navy'); axes[2,3].set_title('CE Loss')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'training_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. VSN Group Weights

In [ ]:
if USE_VSN and model.vsn is not None:
    model.eval()
    all_vsn = []
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch[0].to(DEVICE), batch[1].to(DEVICE))
            if 'vsn_weights' in out:
                all_vsn.append(out['vsn_weights'].cpu().numpy())
    if all_vsn:
        vsn_w = np.concatenate(all_vsn)
        gnames = list(FEATURE_GROUP_SIZES.keys())
        avg = vsn_w.mean(axis=0)
        print(f'{"Group":<15} {"Avg":>8} {"Std":>8}')
        for i, n in enumerate(gnames):
            print(f'{n:<15} {avg[i]:>8.4f} {vsn_w[:,i].std():>8.4f}')

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].bar(gnames, avg, color=plt.cm.Set2(np.linspace(0,1,len(gnames))), edgecolor='black')
        axes[0].axhline(1/len(gnames), color='red', ls='--'); axes[0].set_title('VSN Weights')
        axes[0].tick_params(axis='x', rotation=45)
        axes[1].violinplot([vsn_w[:,i] for i in range(len(gnames))],
                           positions=range(len(gnames)), showmeans=True)
        axes[1].set_xticks(range(len(gnames))); axes[1].set_xticklabels(gnames, rotation=45)
        axes[1].set_title('Weight Distribution')
        plt.tight_layout(); plt.savefig(SAVE_DIR / 'vsn_weights.png', dpi=150); plt.show()
else:
    print('VSN disabled')

## 14. Backtest

In [ ]:
def evaluate_split(ds, loader, name):
    model.eval()
    preds_ret, actuals_ret, positions, pred_classes, actual_classes = [], [], [], [], []
    with torch.no_grad():
        for batch in loader:
            if len(batch) == 5:
                x_seq, ae_in, y_ret, y_std, y_cls = batch
            else:
                x_seq, ae_in, y_ret, y_std = batch; y_cls = None
            out = model(x_seq.to(DEVICE), ae_in.to(DEVICE))
            preds_ret.append(out['pred_return'].cpu().numpy())
            actuals_ret.append(y_ret.numpy())
            if 'class_logits' in out:
                pred_classes.append(out['class_logits'].argmax(dim=-1).cpu().numpy())
            if y_cls is not None: actual_classes.append(y_cls.numpy())
            if 'position' in out: positions.append(out['position'].cpu().numpy())

    pr, ar = np.concatenate(preds_ret), np.concatenate(actuals_ret)
    dates = [ds.get_date(i) for i in range(len(ds))]
    dm = np.abs(ar) > 1e-6
    print(f'\n=== {name} ({len(ds)} samples) ===')
    print(f'  MAE={np.abs(pr-ar).mean():.5f}  Corr={np.corrcoef(pr,ar)[0,1]:.4f}  '
          f'Dir={( np.sign(pr[dm])==np.sign(ar[dm]) ).mean():.3f}')

    if pred_classes and actual_classes:
        pc, ac = np.concatenate(pred_classes), np.concatenate(actual_classes)
        print(f'  Class acc: {(pc==ac).mean():.3f}')

    if positions:
        pos = np.concatenate(positions)
        sr = pos * ar; cum = np.cumsum(sr)
        ann = np.sqrt(252 * bars_per_day)
        sharpe = sr.mean()/sr.std()*ann if sr.std()>0 else 0
        dd = sr[sr<0]; sortino = sr.mean()/dd.std()*ann if len(dd)>1 and dd.std()>0 else 0
        mdd = (cum - np.maximum.accumulate(cum)).min()
        print(f'  Sharpe={sharpe:.3f}  Sortino={sortino:.3f}  MaxDD={mdd:.4f}  TotalRet={cum[-1]:.4f}')
        return {'dates': dates, 'positions': pos, 'strategy_ret': sr, 'cum_ret': cum}
    return {'dates': dates}

val_results = evaluate_split(val_ds, val_loader, 'Validation')
test_results = evaluate_split(test_ds, test_loader, 'Test')

In [ ]:
if 'cum_ret' in test_results:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes[0,0].plot(test_results['dates'], test_results['cum_ret'], color='green')
    axes[0,0].axhline(0, color='black', lw=0.5); axes[0,0].set_title('Test Cumulative Return')
    axes[0,1].hist(test_results['positions'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0,1].set_title('Position Distribution')
    sr = pd.Series(test_results['strategy_ret'])
    rs = sr.rolling(60*bars_per_day).mean()/sr.rolling(60*bars_per_day).std()*np.sqrt(252*bars_per_day)
    axes[1,0].plot(test_results['dates'], rs.values, color='purple')
    axes[1,0].axhline(0, color='black', lw=0.5); axes[1,0].set_title('Rolling Sharpe')
    if 'cum_ret' in val_results:
        axes[1,1].plot(val_results['dates'], val_results['cum_ret'], color='blue', label='Val')
        axes[1,1].plot(test_results['dates'], test_results['cum_ret'], color='green', label='Test')
        axes[1,1].legend(); axes[1,1].set_title('Val vs Test')
    plt.tight_layout()
    plt.savefig(SAVE_DIR / 'backtest.png', dpi=150, bbox_inches='tight'); plt.show()

## 14b. TCN Per-Feature Weight Importance

In [ ]:
# --- TCN input_proj weight importance per feature ---
# When VSN is active, input_proj operates on d_model (VSN output), not raw features.
# In that case, use the VSN group weights + per-group GRN norms for feature importance.

W = model.input_proj.weight.detach().cpu()  # (tcn_ch0, feat_dim)
feat_names = all_feature_cols

if model.vsn is not None:
    # VSN active: input_proj sees d_model, not raw features.
    # Per-group importance = VSN selection weights (from last forward pass aren't stored,
    # so approximate via GRN L1 norms within each group + input_proj norm on d_model).
    print("VSN active — computing per-group GRN-based feature importance")

    # input_proj contribution per d_model dimension
    proj_importance = W.abs().sum(dim=0)  # (d_model,)

    feat_importance = np.zeros(len(feat_names))
    offset = 0
    for gi, (gname, gsize) in enumerate(FEATURE_GROUP_SIZES.items()):
        # First linear in the GRN: (d_model, n_feat_in_group)
        grn_w = model.vsn.group_grns[gi][0].weight.detach().cpu()  # (d_model, gsize)
        # Per raw feature: sum |grn_w| weighted by proj_importance
        raw_imp = (grn_w.abs() * proj_importance.unsqueeze(1)).sum(dim=0).numpy()  # (gsize,)
        feat_importance[offset:offset + gsize] = raw_imp
        offset += gsize

    feat_importance /= feat_importance.sum()
else:
    # No VSN: input_proj directly maps raw features
    assert len(feat_names) == W.shape[1], \
        f"Mismatch: {len(feat_names)} names vs {W.shape[1]} weights"
    feat_importance = W.abs().sum(dim=0).numpy()  # (n_features,)
    feat_importance /= feat_importance.sum()

# Sort descending
order = np.argsort(feat_importance)[::-1]
sorted_names = [feat_names[i] for i in order]
sorted_imp = feat_importance[order]

# Print table
print(f'{"Rank":>4}  {"Feature":<40}  {"Weight":>8}  {"Cumul":>8}')
print('-' * 66)
cumul = 0.0
for rank, (name, imp) in enumerate(zip(sorted_names, sorted_imp), 1):
    cumul += imp
    print(f'{rank:4d}  {name:<40}  {imp:8.4f}  {cumul:8.4f}')

# Plot top-20
top_n = min(20, len(sorted_names))
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart: top features
axes[0].barh(range(top_n), sorted_imp[:top_n][::-1], color='steelblue')
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(sorted_names[:top_n][::-1], fontsize=8)
axes[0].set_xlabel('Normalized L1 Weight')
axes[0].set_title(f'Top-{top_n} TCN Input Feature Importance')

# Group-level aggregation
group_imp = {}
offset = 0
for gname, gsize in FEATURE_GROUP_SIZES.items():
    group_imp[gname] = feat_importance[offset:offset+gsize].sum()
    offset += gsize
gnames = list(group_imp.keys())
gvals = [group_imp[g] for g in gnames]
axes[1].bar(gnames, gvals, color=['#4e79a7','#f28e2b','#e15759','#76b7b2','#59a14f',
                                    '#edc948','#b07aa1'][:len(gnames)])
axes[1].set_ylabel('Group Weight Share')
axes[1].set_title('TCN Feature Importance by Group')
for i, v in enumerate(gvals):
    axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'tcn_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Also compute gradient-based importance on test set for comparison
model.eval()
grad_imp = torch.zeros(len(feat_names))
n_samples = 0
for batch in test_loader:
    if len(batch) == 5:
        x_seq, ae_in, y_ret, y_std, y_cls = batch
    else:
        x_seq, ae_in, y_ret, y_std = batch
    x_seq = x_seq.to(DEVICE).requires_grad_(True)
    ae_in = ae_in.to(DEVICE)
    out = model(x_seq, ae_in)
    out['pred_return'].sum().backward()
    # Mean absolute gradient across batch and time
    grad_imp += x_seq.grad.abs().mean(dim=(0, 1)).cpu()
    n_samples += 1
    model.zero_grad()

grad_imp /= n_samples
grad_imp_np = (grad_imp / grad_imp.sum()).numpy()
grad_order = np.argsort(grad_imp_np)[::-1]

print(f'\n--- Gradient-based importance (top-15) ---')
print(f'{"Rank":>4}  {"Feature":<40}  {"GradImp":>8}  {"WeightImp":>8}')
print('-' * 66)
for rank in range(min(15, len(grad_order))):
    idx = grad_order[rank]
    print(f'{rank+1:4d}  {feat_names[idx]:<40}  {grad_imp_np[idx]:8.4f}  {feat_importance[idx]:8.4f}')

## 15. Save Model

In [ ]:
import json
from dataclasses import asdict

torch.save({
    'model_state_dict': model.state_dict(),
    'config': asdict(final_cfg),
    'best_params': best_params,
    'n_features': n_features,
    'feature_cols': all_feature_cols,
    'regime_cols': REGIME_COLS,
    'feature_groups': {k: list(v) for k, v in feature_groups.items()},
    'bar_minutes': BAR_MINUTES,
    'target_horizon_bars': TARGET_HORIZON_BARS,
    'session': {'name': SESSION.name, 'start': SESSION.start, 'end': SESSION.end},
    'ae_window_bars': ae_window_bars,
}, SAVE_DIR / 'ng_hybrid_intraday_best.pth')

print(f'Saved to {SAVE_DIR / "ng_hybrid_intraday_best.pth"}')
print(f'{BAR_MINUTES}min bars | {SESSION.name} {SESSION.start}-{SESSION.end} | '
      f'horizon={TARGET_HORIZON_BARS} bars | {n_features} features')